# Kalkulator IPK dan Transkrip Nilai

Proyek Akhir — Algoritma dan Pemrograman  
Universitas Al Azhar Indonesia — Semester Genap 2025/2026

| Nama Kelompok           | NIM          |
| :---------------------- | :----------- |
| Achmad Zeehan Wardana   | 0102525001   |
| Akrim Shulhi            | 0102523009   |
| Habibie Rahman Abdillah | 0102525006   |
| Hanif Zakia Khoirunnisa | 0102524642   |
**Kelas:** IF25H
**Dosen:** Tri Aji Nugroho, S.T., M.T.

In [1]:
# Bobot mutu nilai huruf — dict lookup O(1) [Bab 8]
BOBOT_MUTU = {
    'A'  : 4.0,
    'A-' : 3.7,
    'B+' : 3.3,
    'B'  : 3.0,
    'B-' : 2.7,
    'C+' : 2.3,
    'C'  : 2.0,
    'D'  : 1.0,
    'E'  : 0.0,
}

# Daftar nilai huruf yang valid — set untuk validasi O(1) [Bab 8]
NILAI_VALID = set(BOBOT_MUTU.keys())

# Kode mata kuliah yang sudah diinput — set untuk cek duplikat O(1) [Bab 8]
kode_terdaftar = set()

# Riwayat semua mata kuliah — list of dict [Bab 7-8]
# Setiap dict: {"kode", "nama", "sks", "nilai", "bobot_mutu", "semester"}
riwayat_mk = []

# Konstanta tampilan
SEPARATOR = "=" * 60
GARIS     = "-" * 60


# ============================================================
# FUNGSI 1: input_valid — Validasi Input [Bab 3, 4, 5]
# Kompleksitas: O(1) per panggilan (loop hanya karena input ulang)
# ============================================================

In [2]:
def input_valid(prompt, tipe="str", boleh_kosong=False):
    """
    Meminta input dari pengguna dengan validasi tipe data.

    Parameters:
        prompt (str)       : Teks yang ditampilkan ke pengguna.
        tipe (str)         : Tipe data yang diharapkan: 'str', 'int', 'float'.
        boleh_kosong (bool): Apakah input boleh dikosongkan.

    Returns:
        str | int | float  : Nilai input yang sudah divalidasi.
    """
    while True:
        nilai = input(prompt).strip()

        if not nilai and not boleh_kosong:
            print("  ⚠ Input tidak boleh kosong. Coba lagi.")
            continue

        if tipe == "int":
            try:
                hasil = int(nilai)
                return hasil
            except ValueError:
                print("  ⚠ Masukkan angka bulat yang valid. Coba lagi.")
                continue

        if tipe == "float":
            try:
                hasil = float(nilai)
                return hasil
            except ValueError:
                print("  ⚠ Masukkan angka desimal yang valid. Coba lagi.")
                continue

        return nilai


# ============================================================
# FUNGSI 2: tambah_mk — Input Mata Kuliah [FR-1]
# Kompleksitas: O(1) amortisasi (set lookup & list append)
# ============================================================

In [3]:
def tambah_mk():
    """
    Menambahkan data mata kuliah baru ke riwayat dengan validasi.
    Mencakup: validasi kode duplikat (set), nilai huruf (dict lookup),
    dan SKS (range 1-6 sesuai standar akademik UAI).
    """
    print(f"{GARIS}")
    print("  TAMBAH MATA KULIAH BARU")
    print(GARIS)

    # Validasi kode mata kuliah
    while True:
        kode = input_valid("  Kode MK    (contoh: IF1001) : ").upper()
        if kode in kode_terdaftar:
            print(f"  ⚠ Kode '{kode}' sudah terdaftar. Gunakan kode lain.")
        else:
            break

    nama = input_valid("  Nama MK                     : ")

    # Validasi SKS (1–6)
    while True:
        sks = input_valid("  SKS        (1-6)            : ", tipe="int")
        if 1 <= sks <= 6:
            break
        print("  ⚠ SKS harus antara 1 sampai 6.")

    # Validasi nilai huruf
    print(f"Nilai huruf yang valid: {', '.join(sorted(NILAI_VALID))}")
    while True:
        nilai = input_valid("  Nilai Huruf                 : ").upper()
        if nilai in NILAI_VALID:
            break
        print(f"  ⚠ Nilai '{nilai}' tidak valid. Pilih dari daftar di atas.")

    # Validasi semester (1–14 untuk memenuhi standar akademik)
    while True:
        semester = input_valid("  Semester   (1-14)           : ", tipe="int")
        if 1 <= semester <= 14:
            break
        print("  ⚠ Semester harus antara 1 dan 14.")

    # Bangun entri mata kuliah [Bab 8 — Dictionary]
    entri = {
        "kode"       : kode,
        "nama"       : nama,
        "sks"        : sks,
        "nilai"      : nilai,
        "bobot_mutu" : BOBOT_MUTU[nilai],
        "semester"   : semester,
    }

    riwayat_mk.append(entri)     # O(1) amortisasi
    kode_terdaftar.add(kode)     # O(1) — set [Bab 8]

    print(f"✓ Mata kuliah '{nama}' ({kode}) berhasil ditambahkan!")
    print(f"    Nilai: {nilai} | Bobot Mutu: {BOBOT_MUTU[nilai]:.1f} | SKS: {sks} | Semester: {semester}")


# ============================================================
# FUNGSI 3: hitung_ipk — Hitung IPK Kumulatif [FR-2]
# Kompleksitas: O(n) — iterasi seluruh mata kuliah
# ============================================================

In [4]:
def hitung_ipk(daftar_mk=None):
    """
    Menghitung Indeks Prestasi Kumulatif (IPK) menggunakan
    rata-rata berbobot SKS dari seluruh mata kuliah.

    Parameters:
        daftar_mk (list | None): Daftar mata kuliah. Jika None,
                                  gunakan riwayat_mk global.

    Returns:
        float: Nilai IPK dibulatkan 2 desimal. 0.00 jika belum ada data.
    """
    if daftar_mk is None:
        daftar_mk = riwayat_mk

    total_bobot = 0.0   # Σ (SKS × bobot_mutu)
    total_sks   = 0     # Σ SKS

    # O(n) — rata-rata berbobot [Bab 2-4]
    for mk in daftar_mk:
        total_bobot += mk["sks"] * mk["bobot_mutu"]
        total_sks   += mk["sks"]

    if total_sks == 0:
        return 0.00

    return round(total_bobot / total_sks, 2)


# ============================================================
# FUNGSI 4: hitung_ips — Hitung IPS per Semester [FR-2]
# Kompleksitas: O(n) — filter + hitung
# ============================================================

In [5]:
def hitung_ips(semester):
    """
    Menghitung Indeks Prestasi Semester (IPS) untuk satu semester.

    Parameters:
        semester (int): Nomor semester yang ingin dihitung.

    Returns:
        float: Nilai IPS dibulatkan 2 desimal.
    """
    # Filter mata kuliah semester tertentu — O(n) [Bab 4, 7]
    mk_semester = [mk for mk in riwayat_mk if mk["semester"] == semester]
    return hitung_ipk(mk_semester)


# ============================================================
# FUNGSI 5: tampilkan_transkrip — Transkrip Nilai [FR-3]
# Kompleksitas: O(n log n) karena sorting
# ============================================================

In [6]:
def tampilkan_transkrip():
    """
    Menampilkan transkrip nilai yang dikelompokkan per semester
    secara terformat rapi, lengkap dengan IPS dan IPK kumulatif.
    """
    if not riwayat_mk:
        print("⚠ Belum ada data mata kuliah.")
        return

    print(f"{SEPARATOR}")
    print("  TRANSKRIP NILAI AKADEMIK")
    print(f"  Nama: Achmad Zeehan Wardana / Hanif Zakia Khoirunnisa")
    print(f"  NIM : 0102525001 / 0102524642")
    print(SEPARATOR)

    # Kumpulkan daftar semester unik, urutkan [Bab 10 — Sorting]
    semester_list = sorted(set(mk["semester"] for mk in riwayat_mk))

    total_sks_kumulatif = 0
    ipk_kumulatif_mk    = []   # akumulasi untuk hitung IPK seiring semester

    for sem in semester_list:
        # Filter MK semester ini — O(n) [Bab 4]
        mk_sem = [mk for mk in riwayat_mk if mk["semester"] == sem]

        # Urutkan berdasarkan nama MK — O(k log k) [Bab 10]
        mk_sem_sorted = sorted(mk_sem, key=lambda x: x["nama"].lower())

        ips       = hitung_ips(sem)
        sks_sem   = sum(mk["sks"] for mk in mk_sem)
        total_sks_kumulatif += sks_sem
        ipk_kumulatif_mk.extend(mk_sem)
        ipk_sem = hitung_ipk(ipk_kumulatif_mk)

        print(f"╔{'═'*56}╗")
        print(f"  ║  Semester {sem:<47}║")
        print(f"  ╠{'═'*56}╣")
        print(f"  ║  {'Kode':<10} {'Nama Mata Kuliah':<28} {'SKS':>3} {'Nilai':>5} {'Mutu':>5}  ║")
        print(f"  ╠{'-'*56}╣")

        for mk in mk_sem_sorted:
            kode  = mk["kode"][:9]
            nama  = mk["nama"][:27]
            sks   = mk["sks"]
            nilai = mk["nilai"]
            mutu  = mk["bobot_mutu"]
            print(f"  ║  {kode:<10} {nama:<28} {sks:>3} {nilai:>5} {mutu:>5.1f}  ║")

        print(f"  ╠{'-'*56}╣")
        print(f"  ║  Total SKS Semester : {sks_sem:<5}  IPS Semester : {ips:.2f}         ║")
        print(f"  ║  Total SKS Kumulatif: {total_sks_kumulatif:<5}  IPK s.d. Smt ini: {ipk_sem:.2f}  ║")
        print(f"  ╚{'═'*56}╝")

    ipk = hitung_ipk()
    print(f"{'='*30}")
    print(f"  IPK KUMULATIF AKHIR : {ipk:.2f}")
    print(f"  TOTAL SKS LULUS     : {total_sks_kumulatif}")
    print(f"  {'='*30}")

    # Kualifikasi IPK
    if ipk >= 3.51:
        predikat = "Cum Laude"
    elif ipk >= 3.01:
        predikat = "Sangat Memuaskan"
    elif ipk >= 2.76:
        predikat = "Memuaskan"
    elif ipk >= 2.00:
        predikat = "Cukup"
    else:
        predikat = "Perlu Peningkatan"

    print(f"  PREDIKAT            : {predikat}")


# ============================================================
# FUNGSI 6: cari_mk — Pencarian Mata Kuliah [FR-4]
# Algoritma: Linear Search — O(n) [Bab 9]
# ============================================================

In [7]:
def cari_mk():
    """
    Mencari mata kuliah menggunakan linear search berdasarkan
    nama, kode, atau rentang nilai huruf.

    Kompleksitas: O(n) — sesuai pembahasan Big-O di Bab 12.
    """
    if not riwayat_mk:
        print("⚠ Belum ada data mata kuliah.")
        return

    print(f"{GARIS}")
    print("  CARI MATA KULIAH")
    print("  1. Berdasarkan Nama")
    print("  2. Berdasarkan Kode")
    print("  3. Berdasarkan Rentang Nilai (contoh: A, B+)")
    print(GARIS)

    pilihan = input("  Pilih kriteria (1-3): ").strip()

    if pilihan == "1":
        keyword = input_valid("  Kata kunci nama MK: ").lower()
        # Linear search — O(n) [Bab 9]
        hasil = [mk for mk in riwayat_mk if keyword in mk["nama"].lower()]
        label = f"nama mengandung '{keyword}'"

    elif pilihan == "2":
        keyword = input_valid("  Kode MK: ").upper()
        hasil   = [mk for mk in riwayat_mk if keyword in mk["kode"]]
        label   = f"kode mengandung '{keyword}'"

    elif pilihan == "3":
        print(f"  Nilai valid: {', '.join(sorted(NILAI_VALID))}")
        nilai_cari = input_valid("  Masukkan nilai huruf: ").upper()
        if nilai_cari not in NILAI_VALID:
            print("  ⚠ Nilai tidak valid.")
            return
        hasil = [mk for mk in riwayat_mk if mk["nilai"] == nilai_cari]
        label = f"nilai '{nilai_cari}'"

    else:
        print("  ⚠ Pilihan tidak valid.")
        return

    if not hasil:
        print(f"Tidak ditemukan mata kuliah dengan {label}.")
        return

    # Urutkan hasil berdasarkan nama [Bab 10]
    hasil.sort(key=lambda x: x["nama"].lower())

    print(f"Ditemukan {len(hasil)} mata kuliah ({label}):")
    print(f"{'Kode':<10} {'Nama':<28} {'SKS':>3} {'Nilai':>5} {'Mutu':>5}  {'Smt':>4}")
    print(f"  {GARIS}")

    for mk in hasil:
        print(f"  {mk['kode']:<10} {mk['nama'][:27]:<28} {mk['sks']:>3} "
              f"{mk['nilai']:>5} {mk['bobot_mutu']:>5.1f}  {mk['semester']:>4}")


# ============================================================
# FUNGSI 7: urutkan_mk — Pengurutan Transkrip [FR-5]
# Algoritma: Built-in sort Timsort — O(n log n) [Bab 10, 12]
# ============================================================

In [8]:
def urutkan_mk():
    """
    Menampilkan daftar mata kuliah yang diurutkan berdasarkan
    kriteria yang dipilih pengguna menggunakan built-in sort
    (Timsort, O(n log n)).
    """
    if not riwayat_mk:
        print("⚠ Belum ada data mata kuliah.")
        return

    print(f"{GARIS}")
    print("  URUTKAN MATA KULIAH")
    print("  1. Nama (A–Z)")
    print("  2. Nama (Z–A)")
    print("  3. SKS (Terkecil ke Terbesar)")
    print("  4. SKS (Terbesar ke Terkecil)")
    print("  5. Nilai / Bobot Mutu (Tertinggi ke Terendah)")
    print("  6. Nilai / Bobot Mutu (Terendah ke Tertinggi)")
    print("  7. Semester (Awal ke Akhir)")
    print(GARIS)

    pilihan = input("  Pilih kriteria (1-7): ").strip()

    kriteria_map = {
        "1": (lambda x: x["nama"].lower(),          False, "Nama (A–Z)"),
        "2": (lambda x: x["nama"].lower(),           True, "Nama (Z–A)"),
        "3": (lambda x: x["sks"],                   False, "SKS Terkecil"),
        "4": (lambda x: x["sks"],                    True, "SKS Terbesar"),
        "5": (lambda x: x["bobot_mutu"],             True, "Nilai Tertinggi"),
        "6": (lambda x: x["bobot_mutu"],            False, "Nilai Terendah"),
        "7": (lambda x: (x["semester"], x["nama"].lower()), False, "Semester Awal"),
    }

    if pilihan not in kriteria_map:
        print("  ⚠ Pilihan tidak valid.")
        return

    key_func, reverse, label = kriteria_map[pilihan]

    # Timsort built-in — O(n log n) [Bab 10, 12]
    terurut = sorted(riwayat_mk, key=key_func, reverse=reverse)

    print(f"Diurutkan berdasarkan: {label}")
    print(f"{'No':>3}  {'Kode':<10} {'Nama':<28} {'SKS':>3} {'Nilai':>5} {'Mutu':>5}  {'Smt':>4}")
    print(f"  {GARIS}")

    for i, mk in enumerate(terurut, 1):
        print(f"  {i:>3}. {mk['kode']:<10} {mk['nama'][:27]:<28} {mk['sks']:>3} "
              f"{mk['nilai']:>5} {mk['bobot_mutu']:>5.1f}  {mk['semester']:>4}")


# ============================================================
# FUNGSI 8: simulasi_target — Simulasi Target IPK [FR-6]
# Algoritma: Formula balik rata-rata berbobot — O(1) [Bab 2-5]
# ============================================================

In [9]:
def simulasi_target():
    """
    Menghitung bobot mutu minimum yang diperlukan pada semester
    berikutnya agar IPK mencapai target yang diinginkan mahasiswa.

    Formula:
        target_ipk = (total_bobot_sekarang + bobot_baru) / (total_sks + sks_rencana)
        → bobot_baru = target_ipk * (total_sks + sks_rencana) - total_bobot_sekarang
        → mutu_min   = bobot_baru / sks_rencana

    Kompleksitas: O(n) untuk hitung total saat ini, O(1) untuk simulasi.
    """
    if not riwayat_mk:
        print("⚠ Belum ada data mata kuliah. Tambahkan data terlebih dahulu.")
        return

    ipk_sekarang = hitung_ipk()
    total_sks    = sum(mk["sks"] for mk in riwayat_mk)
    total_bobot  = sum(mk["sks"] * mk["bobot_mutu"] for mk in riwayat_mk)

    print(f"{GARIS}")
    print("  SIMULASI TARGET IPK")
    print(GARIS)
    print(f"  IPK Anda saat ini : {ipk_sekarang:.2f}")
    print(f"  Total SKS saat ini: {total_sks}")

    # Validasi target IPK (0.00 – 4.00)
    while True:
        target = input_valid("  Masukkan target IPK yang ingin dicapai (0.00-4.00): ",
                             tipe="float")
        if 0.0 <= target <= 4.0:
            break
        print("  ⚠ Target IPK harus antara 0.00 dan 4.00.")

    # Validasi rencana SKS semester depan
    while True:
        sks_rencana = input_valid("  Rencana jumlah SKS semester depan        : ", tipe="int")
        if 1 <= sks_rencana <= 24:
            break
        print("  ⚠ SKS rencana harus antara 1 dan 24.")

    # Hitung bobot minimum — formula balik O(1)
    bobot_dibutuhkan = target * (total_sks + sks_rencana) - total_bobot
    mutu_min         = bobot_dibutuhkan / sks_rencana if sks_rencana > 0 else 0.0

    print(f"  {'='*40}")
    print(f"  Target IPK           : {target:.2f}")
    print(f"  SKS Rencana          : {sks_rencana}")
    print(f"  Bobot Mutu Minimum   : {mutu_min:.2f}")
    print(f"  {'='*40}")

    # Terjemahkan ke nilai huruf
    if mutu_min <= 0.0:
        print("  ✓ IPK Anda sudah melampaui target! Pertahankan prestasi.")
    elif mutu_min > 4.0:
        print("  ✗ Target IPK tidak dapat dicapai dalam 1 semester dengan SKS tersebut.")
        print(f"    Coba tingkatkan jumlah SKS rencana atau turunkan target sedikit.")
    else:
        # Cari nilai huruf yang sesuai
        nilai_cukup = None
        for nilai_huruf, bobot in sorted(BOBOT_MUTU.items(),
                                         key=lambda x: x[1]):
            if bobot >= mutu_min:
                nilai_cukup = nilai_huruf
                break

        if nilai_cukup:
            print(f"  Nilai minimum yang diperlukan: {nilai_cukup} "
                  f"(Bobot Mutu ≥ {mutu_min:.2f})")
            print(f"  Rekomendasi: Raih nilai {nilai_cukup} atau lebih")
            print(f"  di SEMUA {sks_rencana} SKS yang Anda ambil.")
        else:
            print("  ✗ Target tidak tercapai. Nilai maksimum tidak cukup.")


# ============================================================
# FUNGSI 9: tampilkan_statistik — Statistik Akademik [FR-7]
# Kompleksitas: O(n) untuk semua agregasi
# ============================================================

In [10]:
def tampilkan_statistik():
    """
    Menampilkan ringkasan statistik akademik lengkap meliputi:
    total SKS, distribusi nilai huruf, rata-rata IPK dan IPS,
    serta mata kuliah dengan nilai tertinggi dan terendah.
    """
    if not riwayat_mk:
        print("  ⚠ Belum ada data mata kuliah.")
        return

    print(f"{SEPARATOR}")
    print("  STATISTIK AKADEMIK")
    print(SEPARATOR)

    total_sks   = sum(mk["sks"] for mk in riwayat_mk)   # O(n)
    total_mk    = len(riwayat_mk)
    ipk         = hitung_ipk()

    # Distribusi nilai huruf — dict counter O(n) [Bab 8]
    distribusi = {}
    for mk in riwayat_mk:
        distribusi[mk["nilai"]] = distribusi.get(mk["nilai"], 0) + 1

    # Mata kuliah tertinggi/terendah bobot mutu [Bab 4, 7]
    mk_tertinggi = max(riwayat_mk, key=lambda x: (x["bobot_mutu"], x["sks"]))
    mk_terendah  = min(riwayat_mk, key=lambda x: (x["bobot_mutu"], x["sks"]))

    # IPS per semester
    semester_list = sorted(set(mk["semester"] for mk in riwayat_mk))

    print(f"  Total Mata Kuliah   : {total_mk}")
    print(f"  Total SKS           : {total_sks}")
    print(f"  IPK Kumulatif       : {ipk:.2f}")

    print(f"  --- IPS Per Semester ---")
    for sem in semester_list:
        ips     = hitung_ips(sem)
        mk_sem  = [mk for mk in riwayat_mk if mk["semester"] == sem]
        sks_sem = sum(mk["sks"] for mk in mk_sem)
        print(f"  Semester {sem:>2} : IPS = {ips:.2f}  ({len(mk_sem)} MK, {sks_sem} SKS)")

    print(f"  --- Distribusi Nilai Huruf ---")
    # Urutkan berdasarkan bobot mutu (tertinggi ke terendah)
    for nilai_huruf in sorted(distribusi.keys(),
                              key=lambda x: BOBOT_MUTU[x], reverse=True):
        jumlah  = distribusi[nilai_huruf]
        bar     = "█" * jumlah
        persen  = (jumlah / total_mk) * 100
        print(f"  {nilai_huruf:<3} : {bar:<15} {jumlah:>3} MK ({persen:5.1f}%)")

    print(f"  --- Mata Kuliah Terbaik ---")
    print(f"  {mk_tertinggi['nama']} ({mk_tertinggi['kode']})")
    print(f"  Nilai: {mk_tertinggi['nilai']} | Bobot: {mk_tertinggi['bobot_mutu']:.1f} | SKS: {mk_tertinggi['sks']}")

    print(f"  --- Mata Kuliah Perlu Perhatian ---")
    print(f"  {mk_terendah['nama']} ({mk_terendah['kode']})")
    print(f"  Nilai: {mk_terendah['nilai']} | Bobot: {mk_terendah['bobot_mutu']:.1f} | SKS: {mk_terendah['sks']}")

    # Persentase nilai A dan A- (prestasi tinggi)
    mk_a    = sum(1 for mk in riwayat_mk if mk["nilai"] in ("A", "A-"))
    persen_a = (mk_a / total_mk) * 100 if total_mk > 0 else 0
    print(f"  Persentase nilai A/A- : {persen_a:.1f}% ({mk_a} dari {total_mk} MK)")


# ============================================================
# FUNGSI 10: hapus_mk — Hapus / Edit Mata Kuliah [FR-8]
# Kompleksitas: O(n) — linear search untuk cari kode
# ============================================================

In [11]:
def hapus_mk():
    """
    Menghapus atau mengedit data mata kuliah berdasarkan kode MK
    dengan konfirmasi terlebih dahulu.

    Sub-fitur:
        1. Hapus MK (CRUD — Delete)
        2. Edit nilai huruf MK (CRUD — Update)
    """
    if not riwayat_mk:
        print("  ⚠ Belum ada data mata kuliah.")
        return

    print(f"{GARIS}")
    print("  HAPUS / EDIT MATA KULIAH")
    print("  1. Hapus Mata Kuliah")
    print("  2. Edit Nilai Huruf Mata Kuliah")
    print(GARIS)

    aksi = input("  Pilih aksi (1-2): ").strip()
    if aksi not in ("1", "2"):
        print("  ⚠ Pilihan tidak valid.")
        return

    kode = input_valid("  Masukkan kode MK: ").upper()

    # Linear search — O(n) [Bab 9]
    indeks = None
    for i, mk in enumerate(riwayat_mk):
        if mk["kode"] == kode:
            indeks = i
            break

    if indeks is None:
        print(f"  ⚠ Kode '{kode}' tidak ditemukan.")
        return

    mk_target = riwayat_mk[indeks]
    print(f"  Ditemukan: {mk_target['nama']} ({mk_target['kode']})")
    print(f"  Nilai: {mk_target['nilai']} | SKS: {mk_target['sks']} | Semester: {mk_target['semester']}")

    if aksi == "1":
        konfirmasi = input(f"  Yakin ingin menghapus '{mk_target['nama']}'? (y/n): ").lower()
        if konfirmasi == "y":
            riwayat_mk.pop(indeks)          # O(n) karena geser elemen
            kode_terdaftar.discard(kode)    # O(1) — set
            print(f"  ✓ Mata kuliah '{mk_target['nama']}' berhasil dihapus.")
        else:
            print("  Penghapusan dibatalkan.")

    else:  # aksi == "2"
        print(f"  Nilai valid: {', '.join(sorted(NILAI_VALID))}")
        while True:
            nilai_baru = input_valid("  Masukkan nilai baru: ").upper()
            if nilai_baru in NILAI_VALID:
                break
            print(f"  ⚠ Nilai '{nilai_baru}' tidak valid.")

        nilai_lama = riwayat_mk[indeks]["nilai"]
        riwayat_mk[indeks]["nilai"]      = nilai_baru
        riwayat_mk[indeks]["bobot_mutu"] = BOBOT_MUTU[nilai_baru]

        print(f"  ✓ Nilai '{mk_target['nama']}' diperbarui: {nilai_lama} → {nilai_baru} "
              f"(Mutu: {BOBOT_MUTU[nilai_baru]:.1f})")
        print(f"  IPK baru: {hitung_ipk():.2f}")


# ============================================================
# FUNGSI 11: muat_data_demo — Data Awal untuk Demo / Testing
# ============================================================

In [12]:
def muat_data_demo():
    """
    Memuat data contoh agar program langsung bisa diuji saat demo.
    Berisi 10 mata kuliah dari 2 semester untuk mahasiswa Informatika UAI.
    """
    data_demo = [
        # Semester 1
        {"kode": "UAI1001", "nama": "Pendidikan Agama Islam",      "sks": 2, "nilai": "A",  "bobot_mutu": 4.0, "semester": 1},
        {"kode": "UAI1002", "nama": "Bahasa Indonesia",             "sks": 2, "nilai": "A-", "bobot_mutu": 3.7, "semester": 1},
        {"kode": "IF1001",  "nama": "Algoritma dan Pemrograman",    "sks": 3, "nilai": "A",  "bobot_mutu": 4.0, "semester": 1},
        {"kode": "IF1002",  "nama": "Matematika Diskrit",           "sks": 3, "nilai": "B+", "bobot_mutu": 3.3, "semester": 1},
        {"kode": "IF1003",  "nama": "Pengantar Ilmu Komputer",      "sks": 2, "nilai": "A",  "bobot_mutu": 4.0, "semester": 1},
        # Semester 2
        {"kode": "IF2001",  "nama": "Pemrograman Berorientasi Objek","sks": 3, "nilai": "B+", "bobot_mutu": 3.3, "semester": 2},
        {"kode": "IF2002",  "nama": "Struktur Data",                "sks": 3, "nilai": "B",  "bobot_mutu": 3.0, "semester": 2},
        {"kode": "IF2003",  "nama": "Kalkulus",                     "sks": 3, "nilai": "B-", "bobot_mutu": 2.7, "semester": 2},
        {"kode": "UAI2001", "nama": "Bahasa Inggris",               "sks": 2, "nilai": "A-", "bobot_mutu": 3.7, "semester": 2},
        {"kode": "IF2004",  "nama": "Basis Data",                   "sks": 3, "nilai": "A",  "bobot_mutu": 4.0, "semester": 2},
    ]

    for entri in data_demo:
        riwayat_mk.append(entri)
        kode_terdaftar.add(entri["kode"])

    print("  ✓ Data demo berhasil dimuat (10 mata kuliah, 2 semester).")
    print(f"  IPK awal: {hitung_ipk():.2f}")


# ============================================================
# FUNGSI 12: tampilkan_header & tampilkan_menu — UI Konsol
# ============================================================

In [13]:
def tampilkan_header():
    """Menampilkan header program."""
    print(f"{SEPARATOR}")
    print("     KALKULATOR IPK DAN TRANSKRIP NILAI")
    print("     Universitas Al Azhar Indonesia — 2026")
    print("     Algoritma dan Pemrograman — IF25H")
    print(SEPARATOR)

In [14]:
def tampilkan_menu():
    """Menampilkan menu utama program."""
    ipk_str = f"IPK: {hitung_ipk():.2f}" if riwayat_mk else "IPK: -"
    total_mk = f"{len(riwayat_mk)} MK" if riwayat_mk else "Kosong"

    print(f"  ╔{'═'*48}╗")
    print(f"  ║{'MENU UTAMA':^50}║")
    print(f"  ╠{'═'*48}╣")
    print(f"  ║  Status: {total_mk:<20} {ipk_str:<17}  ║")
    print(f"  ╠{'─'*48}╣")
    print(f"  ║  1. Tambah Mata Kuliah                         ║")
    print(f"  ║  2. Hitung & Tampilkan Transkrip               ║")
    print(f"  ║  3. Hitung IPS per Semester                    ║")
    print(f"  ║  4. Cari Mata Kuliah                           ║")
    print(f"  ║  5. Urutkan Daftar Mata Kuliah                 ║")
    print(f"  ║  6. Simulasi Target IPK                        ║")
    print(f"  ║  7. Statistik Akademik                         ║")
    print(f"  ║  8. Hapus / Edit Mata Kuliah                   ║")
    print(f"  ║  9. Muat Data Demo                             ║")
    print(f"  ║  0. Keluar                                     ║")
    print(f"  ╚{'═'*48}╝")


# ============================================================
# FUNGSI 13: main — Loop Utama Program [Bab 4, 5]
# ============================================================

In [15]:
def main():
    """
    Fungsi utama yang menjalankan loop program dan mengelola
    routing pilihan menu ke fungsi yang sesuai.
    """
    tampilkan_header()
    print("  Selamat datang di Kalkulator IPK dan Transkrip Nilai.")
    print("  Dikembangkan sebagai Proyek Akhir Algoritma dan Pemrograman.")
    print(f"  Tekan '9' untuk muat data demo, atau mulai dengan menu '1'.")

    # Dispatch table — dict of functions O(1) lookup [Bab 8]
    menu_aksi = {
        "1": tambah_mk,
        "2": tampilkan_transkrip,
        "3": menu_ips_semester,
        "4": cari_mk,
        "5": urutkan_mk,
        "6": simulasi_target,
        "7": tampilkan_statistik,
        "8": hapus_mk,
        "9": muat_data_demo,
    }

    # Loop utama — while True [Bab 4]
    while True:
        tampilkan_menu()
        pilihan = input("  Pilih menu (0-9): ").strip()

        if pilihan == "0":
            print(f"  Terima kasih telah menggunakan Kalkulator IPK.")
            print(f"  IPK Akhir Anda: {hitung_ipk():.2f}")
            print(f"  Semangat terus dalam meraih prestasi terbaik!")
            print(f"  Wassalamu'alaikum Warahmatullahi Wabarakatuh.")
            break

        if pilihan in menu_aksi:
            menu_aksi[pilihan]()
        else:
            print("  ⚠ Pilihan tidak valid. Masukkan angka 0–9.")

        input("  Tekan Enter untuk kembali ke menu...")


# ============================================================
# FUNGSI TAMBAHAN: menu_ips_semester — Hitung IPS [FR-2]
# ============================================================

In [16]:
def menu_ips_semester():
    """
    Menu interaktif untuk menghitung IPS semester tertentu
    dan menampilkan daftar mata kuliah pada semester tersebut.
    """
    if not riwayat_mk:
        print("  ⚠ Belum ada data mata kuliah.")
        return

    semester_list = sorted(set(mk["semester"] for mk in riwayat_mk))
    print(f"  Semester tersedia: {', '.join(str(s) for s in semester_list)}")

    while True:
        sem = input_valid("  Masukkan nomor semester: ", tipe="int")
        if sem in semester_list:
            break
        print(f"  ⚠ Semester {sem} tidak ada dalam data.")

    mk_sem  = [mk for mk in riwayat_mk if mk["semester"] == sem]
    ips     = hitung_ips(sem)
    sks_sem = sum(mk["sks"] for mk in mk_sem)

    print(f"{GARIS}")
    print(f"  HASIL IPS — SEMESTER {sem}")
    print(GARIS)
    print(f"  {'Kode':<10} {'Nama':<28} {'SKS':>3} {'Nilai':>5} {'Mutu':>5}")
    print(f"  {GARIS}")

    for mk in sorted(mk_sem, key=lambda x: x["nama"].lower()):
        print(f"  {mk['kode']:<10} {mk['nama'][:27]:<28} {mk['sks']:>3} "
              f"{mk['nilai']:>5} {mk['bobot_mutu']:>5.1f}")

    print(f"  {GARIS}")
    print(f"  Total SKS Semester {sem}: {sks_sem}")
    print(f"  IPS Semester {sem}      : {ips:.2f}")


# ============================================================
# ENTRY POINT
# ============================================================

In [17]:
if __name__ == "__main__":
    main()

     KALKULATOR IPK DAN TRANSKRIP NILAI
     Universitas Al Azhar Indonesia — 2026
     Algoritma dan Pemrograman — IF25H
  Selamat datang di Kalkulator IPK dan Transkrip Nilai.
  Dikembangkan sebagai Proyek Akhir Algoritma dan Pemrograman.
  Tekan '9' untuk muat data demo, atau mulai dengan menu '1'.
  ╔════════════════════════════════════════════════╗
  ║                    MENU UTAMA                    ║
  ╠════════════════════════════════════════════════╣
  ║  Status: Kosong               IPK: -             ║
  ╠────────────────────────────────────────────────╣
  ║  1. Tambah Mata Kuliah                         ║
  ║  2. Hitung & Tampilkan Transkrip               ║
  ║  3. Hitung IPS per Semester                    ║
  ║  4. Cari Mata Kuliah                           ║
  ║  5. Urutkan Daftar Mata Kuliah                 ║
  ║  6. Simulasi Target IPK                        ║
  ║  7. Statistik Akademik                         ║
  ║  8. Hapus / Edit Mata Kuliah                   ║
  ║ 